In [1]:
from pyspark.sql import SparkSession
import os
import sys
import pyarrow
import json
from pyspark.sql.types import FloatType, DateType, StringType,IntegerType, BooleanType
from pyspark.sql.functions import col,create_map, lit,pandas_udf,from_unixtime,when
from shapely.geometry import Point, Polygon
import pandas as pd

In [2]:
jdbc_url = "jdbc:postgresql://localhost:5432/OpenSky"
target_table = 'OpenSky_Aircraft_Bronze'

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += r"C:\hadoop\bin"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable



In [3]:
spark = SparkSession.builder \
    .appName("Test") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .config("spark.driver.memory", "2g") \
    .master("local[*]") \
    .getOrCreate()
    
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
arrow_enabled = spark.conf.get("spark.sql.execution.arrow.pyspark.enabled", "false")

c:\Users\Si3ma\Desktop\spark_simulation\spark_workers_Test\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:

print(f"PyArrow włączony: {arrow_enabled}")
print(f"[INFO] HADOOP_HOME set to: {os.environ['HADOOP_HOME']}")
print(f"[INFO] PYSPARK_PYTHON set to: {os.environ['PYSPARK_PYTHON']}")
print(f"[INFO] PYSPARK_DRIVER_PYTHON set to: {os.environ['PYSPARK_DRIVER_PYTHON']}")
print(f"Zainstalowana wersja PyArrow: {pyarrow.__version__}")
print(spark.version)

PyArrow włączony: true
[INFO] HADOOP_HOME set to: C:\hadoop
[INFO] PYSPARK_PYTHON set to: c:\Users\Si3ma\Desktop\spark_simulation\spark_workers_Test\.venv\Scripts\python.exe
[INFO] PYSPARK_DRIVER_PYTHON set to: c:\Users\Si3ma\Desktop\spark_simulation\spark_workers_Test\.venv\Scripts\python.exe
Zainstalowana wersja PyArrow: 25.0.1
4.2.0


In [5]:

int_list=["time_position","last_contact","position_source","category"]
float_list=["longitude","latitude","geo_altitude","velocity","true_track","vertical_rate","baro_altitude"]
bool_list=["on_ground","spi"]

aircraft_dict = {
    0: "No Info",
    1: "Light",
    2: "Small",
    3: "Medium",
    4: "Large",
    5: "High Vortex Large",
    6: "Heavy",
    7: "High Performance",
    8: "Rotorcraft",
    9: "Glider / Sailplane",
    10: "Lighter than Air",
    11: "Parachutist / Skydiver",
    12: "Ultralight / Hang Glider / Paraglider",
    13: "Reserved / Unassigned",
    14: "Unmanned Aerial Vehicle (UAV)",
    15: "Space / Trans-atmospheric Vehicle",
    16: "Surface Vehicle - Emergency Vehicle",
    17: "Surface Vehicle – Service Vehicle",
    18: "Point Obstacle",
    19: "Cluster Obstacle",
    20: "Line Obstacle"
}

position_source_dict = {
    0: "ADS-B",
    1: "ASTERIX",
    2: "MLAT",
    3: "FLARM"
}


aircraft_map = create_map(
    *[item for kv in aircraft_dict.items() for item in (lit(kv[0]), lit(kv[1]))]
)
position_source_map = create_map(
    *[item for kv in position_source_dict.items() for item in (lit(kv[0]), lit(kv[1]))]
)

with open("poland.json", "r") as f:
    Poland_Polygon = json.load(f)
    
poland_polygon = Polygon(
    Poland_Polygon["features"][0]["geometry"]["coordinates"][0]
)


In [6]:
@pandas_udf("boolean")
def check_point_in_polygon(long: pd.Series, lat: pd.Series) -> pd.Series:
    return pd.Series(
        [
            poland_polygon.contains(Point(lon, la))
            for lon, la in zip(long, lat)
        ]
    )


c:\Users\Si3ma\Desktop\spark_simulation\spark_workers_Test\.venv\Lib\site-packages\pyspark\sql\pandas\functions.py:777: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\Si3ma\Desktop\spark_simulation\spark_workers_Test\.venv\Lib\site-packages\pyspark\sql\pandas\typehints.py:60: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [7]:

with open("db.json", "r") as f:
    connection_properties = json.load(f)

df = spark.read \
    .jdbc(url=jdbc_url, table=target_table, properties=connection_properties)

In [8]:
df.select("ingestion_timestamp").distinct().orderBy("ingestion_timestamp").show(5)

+--------------------+
| ingestion_timestamp|
+--------------------+
|2026-08-14 21:47:...|
|2026-08-15 11:43:...|
+--------------------+



In [9]:

df=df.withColumns({col: df[col].cast(FloatType()) for col in float_list}) \
  .withColumns({col: df[col].cast(IntegerType()) for col in int_list}) \
  .withColumns({col: df[col].cast(BooleanType()) for col in bool_list})

In [10]:
cols=df.columns
cols.remove("ingestion_timestamp")
count_before_enrichment=df.select(cols).distinct().count()

In [11]:

df=df.withColumn("altitude_diff", col("geo_altitude") - col("baro_altitude"))
df=df.withColumn("last_contact_h",from_unixtime(col("last_contact")))
df=df.withColumn("time_position_h",from_unixtime(col("time_position")))
df=df.withColumn(
    "vertical_category",
    when(col("vertical_rate") > 0, "Climbing")
    .otherwise(when(col("vertical_rate")==0,"Constant Altitude")
    .otherwise("Descending"))
    )
df = df.withColumn(
    "aircraft_category",
    aircraft_map.getItem(col("category"))
)
df = df.withColumn(
    "position_source_name",
    position_source_map.getItem(col("position_source"))
)
df = df.withColumn(
    "isPoland",
    check_point_in_polygon(col("longitude"), col("latitude"))
)


c:\Users\Si3ma\Desktop\spark_simulation\spark_workers_Test\.venv\Lib\site-packages\pyspark\sql\classic\column.py:361: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


In [12]:
cols=df.columns
cols.remove("ingestion_timestamp")
count_after_enrichment=df.select(cols).distinct().count()

In [13]:
print(count_before_enrichment==count_after_enrichment)

True


In [55]:
df.select("true_track").orderBy("true_track", ascending=False).show()

+----------+
|true_track|
+----------+
|    359.87|
|    359.87|
|    359.87|
|    359.85|
|    359.84|
|    359.79|
|    359.79|
|    359.77|
|    359.73|
|    359.66|
|    359.62|
|    359.62|
|    359.62|
|    359.61|
|    359.61|
|     359.6|
|    359.58|
|    359.58|
|    359.53|
|    359.52|
+----------+
only showing top 20 rows
